# What this notebook is, and what it found

**Purpose:** MuRIL was the embedding model originally assumed for KCC retrieval
(it's pretrained on Indic languages, so it *sounded* like the right choice).
This notebook tests whether it actually separates topics in practice \u2014 i.e.
whether a wheat-disease query embeds meaningfully differently from an
unrelated query \u2014 and compares it against LaBSE (a model actually trained
for sentence similarity, not just language coverage).

**Method:** embed the same 5,000-chunk stratified KCC sample with both
models, then sample 500 same-topic and 500 different-topic chunk pairs and
compute Cohen's d \u2014 does the model score same-topic pairs meaningfully
higher than different-topic pairs, or not?

## Results (this run)

| | MuRIL | LaBSE |
|---|---|---|
| Cohen's d, grouped by Category | **\u22120.001** (no separation) | 0.047 (negligible) |
| Cohen's d, grouped by QueryType | 0.722 (looks "moderate"...) | 0.719 |
| Same-topic / different-topic mean similarity | 0.9971 / 0.9971 (Category); 0.9977 / 0.9968 (QueryType) | 0.5946 / 0.5899 (Category); 0.6437 / 0.5735 (QueryType) |
| Self-retrieval sanity check | 100% | 100% |

**Bottom line:** MuRIL's embeddings are pretraining-language-aware but not
retrieval-aware \u2014 same-topic and different-topic chunks score almost
identically (0.9971 vs 0.9971), meaning the model can't meaningfully rank
one chunk over another. The "moderate" 0.722 d under QueryType grouping is
misleading on its own: it comes from a *tiny* absolute gap (0.9977 vs
0.9968) inflated by MuRIL's near-zero variance, not real discriminative
power \u2014 see the follow-up cells in this notebook that unpack this
specifically. LaBSE is a real, if modest, improvement (much wider natural
similarity range: std ~0.10 vs MuRIL's ~0.001), confirmed later by held-out
retrieval testing in `06_kcc_retrieval_eval.ipynb`.

**Note:** a third model, `BAAI/bge-m3`, was tested afterward
(`05b_kcc_bge_m3_quick_test.ipynb`) and outperformed both of these by a wide
margin on real retrieval quality \u2014 see that notebook for the full result.
This notebook's value is mainly in documenting *why MuRIL was rejected*,
which independently corroborates the same finding in the team's production
RAG audit (Milestone 3 report, \u00a75.2).


# KCC Embedding & Indexing \u2014 v2: MuRIL vs LaBSE Comparison

**Why v2:** running `05`/`06` (v1) surfaced a real problem, not a code bug \u2014
raw mean-pooled MuRIL embeddings gave near-uniform cosine similarity
(~0.99+) between clearly unrelated chunks (e.g. a wheat-disease query and a
Mentha-planting query scored 0.9976, *higher* than two wheat queries at
0.9963). Qualitative results confirmed it: real queries were returning
topically wrong chunks at high confidence scores.

This is the classic **anisotropy problem** with vanilla, non-fine-tuned
BERT-family sentence embeddings (documented in the original Sentence-BERT
paper, Reimers & Gurevych 2019) \u2014 mean-pooling a model that was never
trained to make "similar meaning \u2192 high cosine similarity" true produces
embeddings that all cluster in a narrow cone of vector space, regardless
of actual content.

**What this notebook does differently:**
1. Embeds the same 5,000-chunk sample with **both** MuRIL (mean-pooled,
   as before) and **LaBSE** (`sentence-transformers/LaBSE`) \u2014 a model
   explicitly trained for cross-lingual sentence similarity, covers Hindi
   and other Indic languages used in the KCC corpus.
2. Runs a **rigorous anisotropy diagnostic**: 500 same-category pairs vs.
   500 different-category pairs per model (not a single spot-check pair),
   with mean/std similarity and a Cohen's d effect size \u2014 a number that
   directly answers "does this model actually separate topics."
3. Builds a separate FAISS index per model so `06` can evaluate both and
   a random-retrieval baseline side by side.


In [1]:
# Step 0: Install dependencies (uncomment on a fresh Colab runtime)
!pip install -q sentence-transformers faiss-cpu transformers torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 38.2 MB/s eta 0:00:00


In [2]:
# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to run on Colab)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import faiss

import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("\u26a0\ufe0f  Running on CPU. Embedding 5,000 chunks with TWO models will take")
    print("   roughly 2x the ~30min seen in v1 for MuRIL alone. Switch to a GPU")
    print("   runtime (Runtime > Change runtime type) if you can \u2014 strongly")
    print("   recommended once you move to the full corpus in a later milestone.")


Using device: cpu
⚠️  Running on CPU. Embedding 5,000 chunks with TWO models will take
   roughly 2x the ~30min seen in v1 for MuRIL alone. Switch to a GPU
   runtime (Runtime > Change runtime type) if you can — strongly
   recommended once you move to the full corpus in a later milestone.


## Step 1: Load Chunk Sample (same stratified sample as v1)

In [4]:
# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to use Drive)
BASE_PATH = "/content/drive/MyDrive/kcc_raw/"

PROCESSED_PATH = f"{BASE_PATH}processed/"
FINAL_PATH = f"{BASE_PATH}final/"

CHUNKS_PATH = f"{PROCESSED_PATH}kcc_chunks_sample_5000.jsonl"

if not Path(CHUNKS_PATH).exists():
    raise FileNotFoundError(
        f"\u274c '{CHUNKS_PATH}' not found. Run 04_kcc_preprocessing.ipynb first."
    )

chunks = []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

texts = [c['text'] for c in chunks]
meta = [c['metadata'] for c in chunks]
print(f"Loaded {len(chunks):,} chunks from: {CHUNKS_PATH}")


Loaded 5,000 chunks from: /content/drive/MyDrive/kcc_raw/processed/kcc_chunks_sample_5000.jsonl


## Step 2: Define Both Embedding Functions

- **MuRIL**: same mean-pooled recipe as v1 (kept for the comparison to be
  apples-to-apples with what you already measured).
- **LaBSE**: `sentence-transformers` handles pooling internally, already
  outputs normalized 768-dim sentence embeddings \u2014 no custom pooling code
  needed, which is itself part of why it tends to behave better out of
  the box.

In [5]:
# --- MuRIL (mean-pooled, same as v1) ---
MURIL_NAME = "google/muril-base-cased"
print(f"Loading {MURIL_NAME} ...")
muril_tokenizer = AutoTokenizer.from_pretrained(MURIL_NAME)
muril_model = AutoModel.from_pretrained(MURIL_NAME).to(DEVICE)
muril_model.eval()
MURIL_DIM = muril_model.config.hidden_size
print(f"\u2705 MuRIL loaded, dim={MURIL_DIM}")


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_muril(texts_batch, batch_size=32, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts_batch), batch_size):
        batch = texts_batch[i:i + batch_size]
        enc = muril_tokenizer(batch, padding=True, truncation=True,
                               max_length=max_length, return_tensors="pt").to(DEVICE)
        out = muril_model(**enc)
        pooled = mean_pooling(out.last_hidden_state, enc["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu().numpy())
    return np.vstack(all_embeddings)


Loading google/muril-base-cased ...


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ MuRIL loaded, dim=768


In [6]:
# --- LaBSE ---
LABSE_NAME = "sentence-transformers/LaBSE"
print(f"Loading {LABSE_NAME} ...")
labse_model = SentenceTransformer(LABSE_NAME, device=DEVICE)
LABSE_DIM = labse_model.get_sentence_embedding_dimension()
print(f"\u2705 LaBSE loaded, dim={LABSE_DIM}")


def embed_labse(texts_batch, batch_size=32):
    return labse_model.encode(
        texts_batch, batch_size=batch_size,
        normalize_embeddings=True, show_progress_bar=False
    )


Loading sentence-transformers/LaBSE ...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

✅ LaBSE loaded, dim=768


## Step 3: Embed All Chunks With Both Models

In [7]:
print(f"Embedding {len(texts):,} chunks with MuRIL...")
t0 = time.time()
muril_embeddings = embed_muril(texts, batch_size=32)
muril_time = time.time() - t0
print(f"\u2705 MuRIL done in {muril_time:.1f}s ({len(texts)/muril_time:.1f} chunks/sec)")

print(f"\nEmbedding {len(texts):,} chunks with LaBSE...")
t0 = time.time()
labse_embeddings = embed_labse(texts, batch_size=32)
labse_time = time.time() - t0
print(f"\u2705 LaBSE done in {labse_time:.1f}s ({len(texts)/labse_time:.1f} chunks/sec)")

print(f"\nMuRIL shape: {muril_embeddings.shape}  |  LaBSE shape: {labse_embeddings.shape}")


Embedding 5,000 chunks with MuRIL...
✅ MuRIL done in 2058.9s (2.4 chunks/sec)

Embedding 5,000 chunks with LaBSE...
✅ LaBSE done in 1301.4s (3.8 chunks/sec)

MuRIL shape: (5000, 768)  |  LaBSE shape: (5000, 768)


## Step 4: Rigorous Anisotropy Diagnostic

v1 checked a single same-category pair vs. a single different-category
pair \u2014 not enough to draw a conclusion from, which is exactly why it
looked fine on paper but fell apart in practice. This version samples
500 pairs of each type and reports:

- **Mean similarity** for same-category vs. different-category pairs
- **Cohen's d** \u2014 the effect size, i.e. how many standard deviations
  apart the two distributions are. d \u2248 0 means the model cannot tell
  same-category from different-category pairs apart *at all* (which is
  what v1's spot-check pair suggested for MuRIL). d > 0.5 is a
  meaningful separation; d > 0.8 is a strong one.

In [8]:
def sample_pairs(n_pairs, same_category, random_state):
    rng = np.random.default_rng(random_state)
    pairs = []
    attempts = 0
    while len(pairs) < n_pairs and attempts < n_pairs * 50:
        i, j = rng.integers(0, len(chunks), size=2)
        attempts += 1
        if i == j:
            continue
        is_same = meta[i]['category'] == meta[j]['category']
        if is_same == same_category:
            pairs.append((i, j))
    return pairs


def cohens_d(a, b):
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else 0.0


N_PAIRS = 500
same_pairs = sample_pairs(N_PAIRS, same_category=True, random_state=42)
diff_pairs = sample_pairs(N_PAIRS, same_category=False, random_state=43)
print(f"Sampled {len(same_pairs)} same-category and {len(diff_pairs)} different-category pairs.")


def diagnose(embeddings, model_name):
    same_sims = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in same_pairs])
    diff_sims = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in diff_pairs])
    d = cohens_d(same_sims, diff_sims)

    print(f"\n{model_name}")
    print("-" * 50)
    print(f"  Same-category similarity:      mean={same_sims.mean():.4f}  std={same_sims.std():.4f}")
    print(f"  Different-category similarity: mean={diff_sims.mean():.4f}  std={diff_sims.std():.4f}")
    print(f"  Separation (same - diff):      {same_sims.mean() - diff_sims.mean():+.4f}")
    print(f"  Cohen's d (effect size):       {d:.3f}", end="  ")
    if d < 0.2:
        print("\u2192 negligible separation (embeddings are ~uninformative for topic)")
    elif d < 0.5:
        print("\u2192 small separation")
    elif d < 0.8:
        print("\u2192 moderate separation")
    else:
        print("\u2192 strong separation")

    return {"same_mean": float(same_sims.mean()), "same_std": float(same_sims.std()),
            "diff_mean": float(diff_sims.mean()), "diff_std": float(diff_sims.std()),
            "cohens_d": float(d)}


muril_diag = diagnose(muril_embeddings, "MuRIL (mean-pooled, v1 recipe)")
labse_diag = diagnose(labse_embeddings, "LaBSE (sentence-transformers)")


Sampled 500 same-category and 500 different-category pairs.

MuRIL (mean-pooled, v1 recipe)
--------------------------------------------------
  Same-category similarity:      mean=0.9971  std=0.0013
  Different-category similarity: mean=0.9971  std=0.0013
  Separation (same - diff):      -0.0000
  Cohen's d (effect size):       -0.001  → negligible separation (embeddings are ~uninformative for topic)

LaBSE (sentence-transformers)
--------------------------------------------------
  Same-category similarity:      mean=0.5946  std=0.1007
  Different-category similarity: mean=0.5899  std=0.0983
  Separation (same - diff):      +0.0047
  Cohen's d (effect size):       0.047  → negligible separation (embeddings are ~uninformative for topic)


## Step 4b: Same Diagnostic, Using `QueryType` Instead of `Category`

`Category` (Cereals, Vegetables, ...) encodes *which crop group* a query is
about, not *what the question is actually asking* \u2014 a wheat-fertilizer
question and a wheat-market-price question share a category despite being
semantically unrelated topics. That's a weak label to test "does this
embedding separate topics" against, and likely part of why LaBSE's Cohen's
d above came back small even though its similarity scores clearly carry
more information than MuRIL's (much wider spread: std 0.10 vs 0.0013).

`QueryType` (Plant Protection, Nutrient Management, Weather-adjacent
advisory, etc. \u2014 see `04_kcc_preprocessing.ipynb`'s metadata schema) is
closer to the actual topic of the question. Rerunning the same 500-pair
diagnostic against `query_type` should give a fairer read on whether LaBSE
is meaningfully separating topics or not.

In [9]:
def sample_pairs_by_field(field, n_pairs, same_group, random_state):
    rng = np.random.default_rng(random_state)
    pairs = []
    attempts = 0
    while len(pairs) < n_pairs and attempts < n_pairs * 50:
        i, j = rng.integers(0, len(chunks), size=2)
        attempts += 1
        if i == j:
            continue
        is_same = meta[i].get(field, 'unknown') == meta[j].get(field, 'unknown')
        if is_same == same_group:
            pairs.append((i, j))
    return pairs


# Sanity check: how many distinct QueryType values are actually in this
# 5,000-chunk sample? A field dominated by one value would make this
# diagnostic as weak as Category was.
from collections import Counter
qtype_dist = Counter(m.get('query_type', 'unknown') for m in meta)
print("QueryType distribution in the sample:")
for qt, count in qtype_dist.most_common(10):
    print(f"  {qt:35s} {count:5d}  ({100*count/len(meta):.1f}%)")

qtype_same_pairs = sample_pairs_by_field('query_type', N_PAIRS, same_group=True, random_state=42)
qtype_diff_pairs = sample_pairs_by_field('query_type', N_PAIRS, same_group=False, random_state=43)
print(f"\nSampled {len(qtype_same_pairs)} same-query_type and {len(qtype_diff_pairs)} different-query_type pairs.")


def diagnose_field(embeddings, model_name, same_pairs, diff_pairs, field_name):
    same_sims = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in same_pairs])
    diff_sims = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in diff_pairs])
    d = cohens_d(same_sims, diff_sims)

    print(f"\n{model_name}  (grouped by {field_name})")
    print("-" * 50)
    print(f"  Same-{field_name} similarity:      mean={same_sims.mean():.4f}  std={same_sims.std():.4f}")
    print(f"  Different-{field_name} similarity: mean={diff_sims.mean():.4f}  std={diff_sims.std():.4f}")
    print(f"  Separation (same - diff):      {same_sims.mean() - diff_sims.mean():+.4f}")
    print(f"  Cohen's d (effect size):       {d:.3f}", end="  ")
    if d < 0.2:
        print("\u2192 negligible separation")
    elif d < 0.5:
        print("\u2192 small separation")
    elif d < 0.8:
        print("\u2192 moderate separation")
    else:
        print("\u2192 strong separation")

    return {"same_mean": float(same_sims.mean()), "same_std": float(same_sims.std()),
            "diff_mean": float(diff_sims.mean()), "diff_std": float(diff_sims.std()),
            "cohens_d": float(d)}


muril_diag_qtype = diagnose_field(muril_embeddings, "MuRIL", qtype_same_pairs, qtype_diff_pairs, "query_type")
labse_diag_qtype = diagnose_field(labse_embeddings, "LaBSE", qtype_same_pairs, qtype_diff_pairs, "query_type")

print("\n" + "=" * 60)
print("COMPARISON: Category-based d vs QueryType-based d")
print("=" * 60)
print(f"{'Model':8s} {'d (Category)':>14s} {'d (QueryType)':>15s}")
print(f"{'MuRIL':8s} {muril_diag['cohens_d']:14.3f} {muril_diag_qtype['cohens_d']:15.3f}")
print(f"{'LaBSE':8s} {labse_diag['cohens_d']:14.3f} {labse_diag_qtype['cohens_d']:15.3f}")
print("\nIf LaBSE's d rises noticeably under QueryType while MuRIL's stays")
print("near zero either way, that confirms: LaBSE does carry real topical")
print("signal, Category was just the wrong axis to measure it on. If both")
print("stay low even under QueryType, that's stronger evidence LaBSE alone")
print("won't be enough and a domain fine-tune is needed before Milestone 4.")


QueryType distribution in the sample:
  Plant Protection                     2305  (46.1%)
  Nutrient Management                   559  (11.2%)
  Fertilizer Use and Availability       526  (10.5%)
  Cultural Practices                    520  (10.4%)
  Weed Management                       303  (6.1%)
  Varieties                             267  (5.3%)
  Seeds and Planting Material           154  (3.1%)
  Water Management                       94  (1.9%)
  Seeds                                  60  (1.2%)
  Field Preparation                      49  (1.0%)

Sampled 500 same-query_type and 500 different-query_type pairs.

MuRIL  (grouped by query_type)
--------------------------------------------------
  Same-query_type similarity:      mean=0.9977  std=0.0011
  Different-query_type similarity: mean=0.9968  std=0.0014
  Separation (same - diff):      +0.0009
  Cohen's d (effect size):       0.722  → moderate separation

LaBSE  (grouped by query_type)
-------------------------------------

## Step 5: Build FAISS Indices (one per model)

In [10]:
muril_index = faiss.IndexFlatIP(MURIL_DIM)
muril_index.add(muril_embeddings.astype('float32'))

labse_index = faiss.IndexFlatIP(LABSE_DIM)
labse_index.add(labse_embeddings.astype('float32'))

print(f"\u2705 MuRIL index: {muril_index.ntotal:,} vectors, dim={muril_index.d}")
print(f"\u2705 LaBSE index: {labse_index.ntotal:,} vectors, dim={labse_index.d}")


✅ MuRIL index: 5,000 vectors, dim=768
✅ LaBSE index: 5,000 vectors, dim=768


In [11]:
# Self-retrieval sanity check for both (should both be ~100% \u2014 this is
# the trivial check; it does NOT distinguish the two models the way the
# anisotropy diagnostic above does)
def self_retrieval_check(index, embeddings, n=100, random_state=1):
    rng = np.random.default_rng(random_state)
    idxs = rng.choice(len(embeddings), size=min(n, len(embeddings)), replace=False)
    hits = 0
    for idx in idxs:
        _, retrieved = index.search(embeddings[idx:idx + 1].astype('float32'), k=1)
        if retrieved[0][0] == idx:
            hits += 1
    return hits / len(idxs)

muril_self = self_retrieval_check(muril_index, muril_embeddings)
labse_self = self_retrieval_check(labse_index, labse_embeddings)
print(f"MuRIL self-retrieval: {muril_self:.1%}")
print(f"LaBSE self-retrieval: {labse_self:.1%}")


MuRIL self-retrieval: 100.0%
LaBSE self-retrieval: 100.0%


## Step 6: Save Artifacts (both models, clearly namespaced)

In [12]:
Path(FINAL_PATH).mkdir(parents=True, exist_ok=True)

faiss.write_index(muril_index, f"{FINAL_PATH}kcc_faiss_index_muril.bin")
faiss.write_index(labse_index, f"{FINAL_PATH}kcc_faiss_index_labse.bin")
np.save(f"{FINAL_PATH}kcc_chunk_embeddings_muril.npy", muril_embeddings)
np.save(f"{FINAL_PATH}kcc_chunk_embeddings_labse.npy", labse_embeddings)

lookup_path = f"{FINAL_PATH}kcc_index_metadata.jsonl"
with open(lookup_path, 'w', encoding='utf-8') as f:
    for i, c in enumerate(chunks):
        f.write(json.dumps({"id": i, "text": c["text"], "metadata": c["metadata"]}, ensure_ascii=False) + "\n")
print(f"Saved: {lookup_path}  (shared by both models \u2014 same chunk order)")

summary = {
    "muril": {"model": MURIL_NAME, "dim": int(MURIL_DIM), "embed_time_sec": round(muril_time, 1),
              "self_retrieval": round(muril_self, 4),
              "diagnostic_by_category": muril_diag,
              "diagnostic_by_query_type": muril_diag_qtype},
    "labse": {"model": LABSE_NAME, "dim": int(LABSE_DIM), "embed_time_sec": round(labse_time, 1),
              "self_retrieval": round(labse_self, 4),
              "diagnostic_by_category": labse_diag,
              "diagnostic_by_query_type": labse_diag_qtype},
    "n_chunks": len(chunks),
    "n_diagnostic_pairs_per_group": N_PAIRS,
    "query_type_distribution_in_sample": dict(qtype_dist.most_common(10)),
}
with open(f"{FINAL_PATH}kcc_embedding_comparison_summary.json", 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print("\n" + json.dumps(summary, indent=2))


Saved: /content/drive/MyDrive/kcc_raw/final/kcc_index_metadata.jsonl  (shared by both models — same chunk order)

{
  "muril": {
    "model": "google/muril-base-cased",
    "dim": 768,
    "embed_time_sec": 2058.9,
    "self_retrieval": 1.0,
    "diagnostic_by_category": {
      "same_mean": 0.9970700740814209,
      "same_std": 0.0013326621847227216,
      "diff_mean": 0.9970710277557373,
      "diff_std": 0.001257210737094283,
      "cohens_d": -0.0007354153785854578
    },
    "diagnostic_by_query_type": {
      "same_mean": 0.9977244734764099,
      "same_std": 0.0010845694923773408,
      "diff_mean": 0.9968301057815552,
      "diff_std": 0.0013720616698265076,
      "cohens_d": 0.7224664092063904
    }
  },
  "labse": {
    "model": "sentence-transformers/LaBSE",
    "dim": 768,
    "embed_time_sec": 1301.4,
    "self_retrieval": 1.0,
    "diagnostic_by_category": {
      "same_mean": 0.5945702791213989,
      "same_std": 0.100654736161232,
      "diff_mean": 0.5899103879928589,


---
## Verdict So Far

If LaBSE's Cohen's d is meaningfully higher than MuRIL's (expect it to
be \u2014 LaBSE was trained for exactly this kind of semantic separation
task, MuRIL wasn't), that's your evidence for switching the architecture
doc's embedding model choice before Milestone 4. The retrieval-quality
comparison (recall@k, held-out category match, and a random-baseline
control) happens in `06_kcc_retrieval_eval.ipynb` \u2014 proceed there next.
